---
title: Flickering Investigations
authors:
  - name: James Butler
    affiliations: ucb
  - name: Michelle Maclennan
    affiliation: bas
  - name: Fernando Pérez
    affiliation: ucb
  - name: Jon McAuliffe
    affiliation: ucb
affiliations:
  - id: ucb
    institution: University of California Berkeley
    ror: https://ror.org/01an7q238
    department: Statistics
  - id: bas
    institution: British Antarctic Survey
    ror: https://ror.org/01rhff309
---

Since ARs are detected in @wille_antarctic_2021 algorithm by excesses of vIVT over pixelwise thresholds (among other geometric criteria), it is possible for ARs to flicker on and off from one time step to the next as their corresponding vIVT values straddle the thresholds. It is also possible for them to drastically change shape (for example, drastically lengthen or shorten) from one time step to the next. This irregularity largely reflects the notorious and inherent difficulty in developing concrete AR definitions. However, it nonetheless creates a challenge for tracking ARs over time. A benefit of using a spatiotemporal density-based clustering method like `ST-DBSCAN` is that, even if the pixels disappear for a relatively small amount of time, it is possible for the pixels reappearing later to nonetheless be stitched to set from earlier due to the temporal neighborhood conditions in the algorithm. However, if an AR flickers off and back on, is it the same AR or a different AR? Will a clustering algorithm be able to get the right answer? Is there even a right answer? 

In this notebook, we showcase some examples of AR tracking where it appears an AR is flickering on and off to examine what our clustering procedure does. We do not attempt to answer the above questions in each flickering case, namely if the AR is the same or not. Our main goal with this notebook is to make the user aware of how the procedure tends to handle such cases, so they can be more informed should they decide to use it for their own data.

In [1]:
import numpy as np
import xarray as xr
from matplotlib import animation
import matplotlib.pyplot as plt
from IPython.display import Video
import pandas as pd

from artools.format_utils import to_stormtime_format
from artools.display_utils import make_movie, make_eulerian_movie, display_catalog, plot_stormtime_grid
from artools.loading_utils import load_catalog, load_wille_catalogs
from datetime import date

from IPython.display import Video

## Loading up the catalogs

We load them into "stormtime format," a tabular dataframe where each row corresponds to a particular AR at a particular time step. This format faciliates the creation of animations.

In [2]:
catalog = load_catalog('epsspace0.5_epstime12_minpts5_nreppts10_seed12345.h5')
landfalling = catalog[catalog.is_landfalling]

stormtime_format = to_stormtime_format(landfalling, show_progress=True)

Processing storms:   0%|          | 0/3165 [00:00<?, ?it/s]

In [4]:
stormtime_format

,label,time,lat,lon
0,1.0,1980-01-02 00:00:00,"[-60.5, -60.5, -60.5, -60.5, -60.5, -60.5, -60...","[-113.75, -113.125, -112.5, -111.875, -111.25,..."
1,1.0,1980-01-02 03:00:00,"[-62.0, -62.0, -62.0, -62.0, -62.0, -62.0, -61...","[-107.5, -106.875, -106.25, -105.625, -105.0, ..."
2,1.0,1980-01-02 06:00:00,"[-63.0, -63.0, -63.0, -63.0, -63.0, -63.0, -63...","[-108.125, -107.5, -106.875, -106.25, -105.625..."
3,1.0,1980-01-02 09:00:00,"[-64.0, -64.0, -64.0, -64.0, -63.5, -63.5, -63...","[-103.75, -103.125, -102.5, -101.875, -106.25,..."
4,1.0,1980-01-02 12:00:00,"[-65.0, -65.0, -65.0, -65.0, -64.5, -64.5, -64...","[-102.5, -101.875, -101.25, -100.625, -103.75,..."
...,...,...,...,...
32855,9076.0,2022-12-30 00:00:00,"[-69.5, -69.5, -69.0, -69.0, -69.0, -69.0, -69...","[38.75, 39.375, 38.75, 39.375, 40.0, 48.75, 49..."
32856,9078.0,2022-12-31 12:00:00,"[-67.0, -66.5, -66.5, -66.5, -66.5, -66.0, -66...","[130.0, 128.125, 128.75, 129.375, 130.0, 131.2..."
32857,9078.0,2022-12-31 15:00:00,"[-67.0, -67.0, -67.0, -67.0, -67.0, -67.0, -66...","[128.125, 128.75, 129.375, 130.0, 130.625, 131..."
32858,9078.0,2022-12-31 18:00:00,"[-67.0, -67.0, -67.0, -67.0, -67.0, -67.0, -67...","[138.75, 139.375, 140.0, 140.625, 141.25, 141...."


## Flickering Examples

### February 1980

In the below animation, we can see an AR 29 approaching East Antarctica. The AR abruptly disappears for 3 hours on February 8 at 12:00 UTC, before reappearing. The clustering algorithm gives it the same label.

In [5]:
ani_name = '021980_flickering.mp4'

start = date(1980, 2, 5)
end = date(1980, 2, 13)

stormtime_subset = stormtime_format[stormtime_format['time'].dt.date.between(start, end, inclusive='both')]
ani = make_movie(stormtime_subset, 'February 1980 Flickering Events', '../../output/animations/' + ani_name)

Video('../../output/animations/' + ani_name, embed=True, html_attributes='controls')

Saving animation to ../../output/animations/021980_flickering.mp4...


  0%|          | 0/30 [00:00<?, ?it/s]

### May 1980

AR 95 makes landfall in Marie Byrd Land, but is last observed on May 25 at 12:00 UTC. 24 hours later, AR 96 appears in the same location. A different label is to be expected as our clustering algorithm only uses a 12-hour time window to search for AR pixels neighboring in time.

In [6]:
ani_name = '051980_flickering.mp4'

start = date(1980, 5, 20)
end = date(1980, 5, 26)

stormtime_subset = stormtime_format[stormtime_format['time'].dt.date.between(start, end, inclusive='both')]
ani = make_movie(stormtime_subset, 'May 1980 Flickering Events', '../../output/animations/' + ani_name)

Video('../../output/animations/' + ani_name, embed=True, html_attributes='controls')

Saving animation to ../../output/animations/051980_flickering.mp4...


  0%|          | 0/22 [00:00<?, ?it/s]

### June 2009

AR 5890 flickers off for 6 hours on June 13, but comes back on with the same label. AR 5893 then approaches, and it appears the mass of AR pixels abruptly changes shape to include a large landfalling component over Queen Maud Land/Enderby Land. This AR is then given a different label (5894). Once this chunk of the AR of AIS disappears 6 hours later, the blob of AR pixels are returned to their old label (5893).

In [7]:
ani_name = '062009_flickering.mp4'

start = date(2009, 6, 13)
end = date(2009, 6, 17)

stormtime_subset = stormtime_format[stormtime_format['time'].dt.date.between(start, end, inclusive='both')]
ani = make_movie(stormtime_subset, 'June 2009 Flickering Events', '../../output/animations/' + ani_name)

Video('../../output/animations/' + ani_name, embed=True, html_attributes='controls')

Saving animation to ../../output/animations/062009_flickering.mp4...


  0%|          | 0/40 [00:00<?, ?it/s]